# RT-DETR_ReID — 탐지 + 보행자 속성 인식 (단일 모델)

동결 RT-DETR가 사람을 탐지하고, 각 사람의 **디코더 query 임베딩**과 **박스 영역 원본 픽셀**(작은 conv, RegionStem)을 함께 보고 속성(나이·성별·상의색·하의색)을 예측한다.
탐지용 query 임베딩만으로는 뭉개지던 **색 정보를 원본 픽셀 조회로 복원** → 단일 모델·단일 forward로 color까지 잡는다.

**위→아래 순서대로 실행**: 마운트 → 이미지추출 → 라벨 파이프라인 → 박스 추출(Phase 1A) → 학습 → 추론.


In [ ]:
# Drive 마운트 + 경로
from google.colab import drive
import os, shutil
if os.path.isdir('/content/drive') and not os.path.ismount('/content/drive'):
    shutil.rmtree('/content/drive', ignore_errors=True)
try:
    drive.mount('/content/drive')
except ValueError:
    drive.mount('/content/drive', force_remount=True)

BASE     = '/content/drive/MyDrive/Workspace/RT-DETR_ReID'
UPAR_DIR = f'{BASE}/dataset/UPAR'
CACHE    = f'{BASE}/query_cache_v0_3'
CKPT_DIR = f'{BASE}/checkpoints/region_attr'
os.makedirs(CKPT_DIR, exist_ok=True)
print('train.csv', os.path.exists(f'{UPAR_DIR}/train.csv'),
      '| cache train.pt', os.path.exists(f'{CACHE}/train.pt'))

In [ ]:
# ============================================================
# 원본 데이터 획득 (Market-1501 + PA-100K) — Drive dataset 폴더로 다운로드
#   키는 본인 것으로 교체. 이미 Drive에 zip/추출본이 있으면 이 셀은 건너뛴다.
# ============================================================
import os, json, zipfile
DATASET_DIR = '/content/drive/MyDrive/Workspace/RT-DETR_ReID/dataset'
os.makedirs(DATASET_DIR, exist_ok=True)

# Market-1501 (Google Drive 공개 미러)
MARKET_DIR = f'{DATASET_DIR}/Market-1501-v15.09.15'
if not os.path.exists(MARKET_DIR):
    !pip install -q gdown
    import gdown
    gdown.download('https://drive.google.com/uc?id=0B8-rUzbwVRk0c054eEozWG9COHM',
                   f'{DATASET_DIR}/Market-1501.zip', quiet=False)
    with zipfile.ZipFile(f'{DATASET_DIR}/Market-1501.zip') as z:
        z.extractall(DATASET_DIR)
else:
    print('Market-1501 이미 존재')

# PA-100K (Kaggle) — 본인 Kaggle 계정의 username/key 입력
PA100K_DIR = f'{DATASET_DIR}/PA-100K'
os.makedirs(PA100K_DIR, exist_ok=True)
if not os.path.exists(f'{PA100K_DIR}/data'):
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'w') as kf:
        json.dump({'username': 'YOUR_KAGGLE_USERNAME', 'key': 'YOUR_KAGGLE_KEY'}, kf)
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    !pip install -q kaggle
    !kaggle datasets download -d yuulind/pa-100k -p {PA100K_DIR} --unzip -q
    print('PA-100K 다운로드 완료')
else:
    print('PA-100K 이미 존재')

In [ ]:
# 크롭 이미지 추출 (런타임마다 1회) — Drive archive zip → /content/dataset
# train.csv 의 /content/dataset/{PA100k,Market}/... 경로를 그대로 복원한다.
import os, zipfile, time, sys
DRIVE = f'{BASE}/dataset'
LOCAL = '/content/dataset'

jobs    = {'PA100k': f'{DRIVE}/archive (1).zip',    # PA-100K
           'Market': f'{DRIVE}/archive (2).zip'}    # Market-1501
MARKERS = {'PA100k': 'data', 'Market': 'bounding_box_train'}

def _have(name, marker):
    base = f'{LOCAL}/{name}'
    if not os.path.isdir(base):
        return False
    for root, dirs, _ in os.walk(base):
        if marker in dirs:
            return True
    return False

def _extract(zp, out, name):
    with zipfile.ZipFile(zp) as z:
        members = z.infolist(); total = len(members); t = time.time()
        for i, m in enumerate(members, 1):
            z.extract(m, out)
            if i % 500 == 0 or i == total:
                el = time.time() - t; eta = el / i * (total - i)
                sys.stdout.write(f'\r  {name}: {i:>6}/{total} ({i/total*100:5.1f}%)  {el:4.0f}s ETA {eta:4.0f}s')
                sys.stdout.flush()
        print()

for name, zp in jobs.items():
    if _have(name, MARKERS[name]):
        print(f'[{name}] 이미 추출됨 → 스킵'); continue
    if not os.path.exists(zp):
        print(f'[{name}] zip 없음: {zp}'); continue
    print(f'[{name}] 해제 시작: {zp}')
    _extract(zp, f'{LOCAL}/{name}', name)
print('추출 완료 → /content/dataset 준비됨')


## 데이터 라벨 파이프라인 (메인 노트북과 동일 처리)
UPAR pkl 다운로드 → 색/속성 변환 → merge(절대경로 csv) → train/val 분할. (이미 있으면 빠르게 통과)

In [ ]:
import os
DATASET_DIR = '/content/drive/MyDrive/Workspace/RT-DETR_ReID/dataset'
UPAR_DIR    = f'{DATASET_DIR}/UPAR'
LOCAL       = '/content/dataset'
os.makedirs(UPAR_DIR, exist_ok=True)

# UPAR 어노테이션(dataset_all.pkl)만 받는다.
#   이미지 데이터(Market/PA-100K)는 앞의 '로컬 추출' 셀이
#   archive*.zip(Drive) -> /content/dataset 로 푼다. (Drive 직접 해제는 I/O가 느려 제거)
if not os.path.exists(f'{UPAR_DIR}/upar_dataset'):
    !git clone https://github.com/speckean/upar_dataset.git {UPAR_DIR}/upar_dataset
else:
    print('UPAR 어노테이션 이미 존재')

# 어노테이션 pkl 존재 확인
pkl = f'{UPAR_DIR}/upar_dataset/UPAR/dataset_all.pkl'
print('dataset_all.pkl:', 'OK' if os.path.exists(pkl) else '없음')

# 로컬 추출본(이미지) 점검 — 없으면 '로컬 추출' 셀을 먼저 실행
try:
    _pm = {k: PREFIX_MAP[k] for k in ('Market1501', 'PA100k') if k in PREFIX_MAP}
except NameError:
    _pm = {}
checks = [('Market',  _pm.get('Market1501', f'{LOCAL}/Market/Market-1501-v15.09.15')),
          ('PA-100K', _pm.get('PA100k',     f'{LOCAL}/PA100k/PA-100K'))]
missing = False
for name, p in checks:
    ok = os.path.exists(p)
    missing = missing or not ok
    print(f'  [{name}] {p}: {"OK" if ok else "없음"}')
if missing:
    print('\n[안내] 이미지 추출본이 없습니다. 위의 "로컬 추출" 셀(archive*.zip -> /content)을 먼저 실행하세요.')
else:
    print('\n소스 데이터셋 준비 완료')


In [ ]:
import pickle, os
import numpy as np

DATASET_DIR = '/content/drive/MyDrive/Workspace/RT-DETR_ReID/dataset'
UPAR_DIR    = f'{DATASET_DIR}/UPAR'

# UPAR 40개 속성 그룹 인덱스 (speckean/upar_dataset WACV2023)
AGE_IDX      = [0, 1, 2]            # Young, Adult, Old
GENDER_IDX   = 3                    # Female (0=male, 1=female)
UP_COLOR_IDX = list(range(8, 21))   # 13색
DN_COLOR_IDX = list(range(22, 35))  # 13색
# 색상 순서: black,blue,brown,green,grey,orange,pink,purple,red,white,yellow,mixture,other

def binary_to_group(attrs, indices):
    for j, i in enumerate(indices):
        if int(attrs[i]) == 1:
            return j
    return len(indices) - 1  # other

def upar_to_label(attrs):
    age      = int(np.argmax([attrs[i] for i in AGE_IDX]))
    gender   = int(attrs[GENDER_IDX])
    up_color = binary_to_group(attrs, UP_COLOR_IDX)
    dn_color = binary_to_group(attrs, DN_COLOR_IDX)
    return age, gender, up_color, dn_color

pkl_path = f'{UPAR_DIR}/upar_dataset/UPAR/dataset_all.pkl'
with open(pkl_path, 'rb') as f:
    upar_data = pickle.load(f)

print(f'UPAR 로드: {len(upar_data)}개')
sample = upar_data[0] if isinstance(upar_data, list) else upar_data
print('샘플 키:', list(sample.keys()) if isinstance(sample, dict) else type(sample))


In [ ]:
import pickle, os
import pandas as pd
import numpy as np

DATASET_DIR = '/content/drive/MyDrive/Workspace/RT-DETR_ReID/dataset'
UPAR_DIR    = f'{DATASET_DIR}/UPAR'
OUT_CSV     = f'{UPAR_DIR}/upar_merged.csv'

# 이미지는 /content 로컬(빠름) 추출본 사용 (앞 추출 셀 선행).
# 백본 캐시 생성용으로만 쓰고 캐시(.pt)는 Drive에 영구 저장.
LOCAL = '/content/dataset'
# /content 추출본에서 직접 root 자동탐지 (셀 실행 순서 무관, 압축 nesting에 견고).
# PETA/RAP2는 제외(원본 구조 불일치).
_MARKERS = {'Market1501': ('Market', 'bounding_box_train'),
            'PA100k':     ('PA100k', 'data')}
def _find_root(base, marker):
    if not os.path.isdir(base):
        return None
    for root, dirs, _ in os.walk(base):
        if marker in dirs:
            return root
    return None
PREFIX_MAP = {}
for _key, (_sub, _marker) in _MARKERS.items():
    _r = _find_root(f'{LOCAL}/{_sub}', _marker)
    if _r:
        PREFIX_MAP[_key] = _r
    print(f'  {_key}: {_r}')
if not PREFIX_MAP:
    # 자동 추출 폴백: Drive archive → /content(로컬 SSD, 빠름). 순서 의존 제거.
    import zipfile
    DRIVE = '/content/drive/MyDrive/Workspace/RT-DETR_ReID/dataset'
    _auto = {'PA100k': f'{DRIVE}/archive (1).zip', 'Market': f'{DRIVE}/archive (2).zip'}
    print('추출본 없음 → Drive archive에서 자동 추출 시도...')
    for _nm, _zp in _auto.items():
        if os.path.exists(_zp):
            print(f'  [{_nm}] 해제: {_zp}')
            with zipfile.ZipFile(_zp) as _z:
                _z.extractall(f'{LOCAL}/{_nm}')
        else:
            print(f'  [{_nm}] zip 없음: {_zp}')
    for _key, (_sub, _marker) in _MARKERS.items():
        _r = _find_root(f'{LOCAL}/{_sub}', _marker)
        if _r:
            PREFIX_MAP[_key] = _r
        print(f'  {_key}: {_r}')
if not PREFIX_MAP:
    raise RuntimeError("이미지 추출본/zip을 찾지 못했습니다. Drive의 dataset 폴더에 'archive (1).zip'(PA-100K), 'archive (2).zip'(Market-1501)을 두세요.")
print('PREFIX_MAP =', PREFIX_MAP)

pkl_path = f'{UPAR_DIR}/upar_dataset/UPAR/dataset_all.pkl'
with open(pkl_path, 'rb') as f:
    upar_data = pickle.load(f)

img_names = upar_data.image_name
labels    = upar_data.label
print('N images:', len(img_names))

records, skipped = [], 0
for idx in range(len(img_names)):
    rel   = str(img_names[idx])
    attrs = labels[idx]
    prefix = rel.split('/')[0]
    if prefix not in PREFIX_MAP:
        skipped += 1; continue
    rel_rest = rel[len(prefix) + 1:]
    abs_path = os.path.join(PREFIX_MAP[prefix], rel_rest)
    if not os.path.exists(abs_path):
        skipped += 1; continue
    age, gender, up_color, dn_color = upar_to_label(attrs)
    records.append({
        'filepath': abs_path, 'dataset': prefix,
        'age': age, 'gender': gender,
        'up_color': up_color, 'down_color': dn_color,
    })

df = pd.DataFrame(records)
df.to_csv(OUT_CSV, index=False)
print(f'유효: {len(df)}  스킵: {skipped}')
if len(df):
    print(df['dataset'].value_counts())
else:
    print('\n[진단] 유효 0 → 경로/파일명 불일치 점검')
    seen = set()
    for idx in range(len(img_names)):
        rel = str(img_names[idx]); pref = rel.split('/')[0]
        if pref in PREFIX_MAP and pref not in seen:
            seen.add(pref)
            rest = rel[len(pref)+1:]
            ap = os.path.join(PREFIX_MAP[pref], rest)
            d = os.path.dirname(ap)
            print(f'  [{pref}] rel={rel}')
            print(f'         기대={ap}  exists={os.path.exists(ap)}')
            print(f'         디렉토리={d}  exists={os.path.isdir(d)}')
            if os.path.isdir(d):
                print(f'         실제파일 예시={os.listdir(d)[:3]}')
        if len(seen) >= len(PREFIX_MAP):
            break


In [ ]:
import pandas as pd, json, numpy as np

DATASET_DIR = '/content/drive/MyDrive/Workspace/RT-DETR_ReID/dataset'
UPAR_DIR    = f'{DATASET_DIR}/UPAR'
df          = pd.read_csv(f'{UPAR_DIR}/upar_merged.csv')

# UPAR partition 대신 자체 랜덤 9:1 분할로 비율 보장
VAL_RATIO = 0.1
rng = np.random.default_rng(42)
idx = rng.permutation(len(df))
n_val = int(len(df) * VAL_RATIO)
val_idx, train_idx = idx[:n_val], idx[n_val:]

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df   = df.iloc[val_idx].reset_index(drop=True)

train_df.to_csv(f'{UPAR_DIR}/train.csv', index=False)
val_df.to_csv(f'{UPAR_DIR}/val.csv',     index=False)

split = {
    'train_imgs':  train_df['filepath'].tolist(),
    'val_imgs':    val_df['filepath'].tolist(),
    'train_attrs': train_df[['age','gender','up_color','down_color']].values.tolist(),
    'val_attrs':   val_df[['age','gender','up_color','down_color']].values.tolist(),
}
with open(f'{UPAR_DIR}/split.json', 'w') as f:
    json.dump(split, f)

print(f'train: {len(train_df)}  /  val: {len(val_df)}  (비율 {len(train_df)/max(len(val_df),1):.1f}:1)')
print('--- train 데이터셋 구성 ---')
print(train_df['dataset'].value_counts())
print('생성 완료')


## 박스 필터 + pseudo-GT 박스 추출 (Phase 1A, 동일 처리)
동결 RT-DETR로 사람 query 박스/임베딩 추출 → query_cache_v0_3 에 저장(이미 있으면 스킵).

In [ ]:
# =============================================================
# Phase 1A: query 임베딩 정밀계산 + 박스 필터 (1회, Drive 영구저장 ~70MB)
#   동결 backbone+decoder로 각 이미지의 "사람 query" 256-d 임베딩 추출
#   사람 score < THRESH 이미지는 제외(= 박스 필터). 결과는 Drive에 저장.
# =============================================================
!pip install ultralytics -q
import os, json, torch, torch.nn as nn, numpy as np, pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
from ultralytics import RTDETR
from ultralytics.nn.modules.transformer import DeformableTransformerDecoder
from ultralytics.nn.modules.utils import inverse_sigmoid

UPAR_DIR     = '/content/drive/MyDrive/Workspace/RT-DETR_ReID/dataset/UPAR'
OUT_DIR      = '/content/drive/MyDrive/Workspace/RT-DETR_ReID/query_cache_v0_3'
os.makedirs(OUT_DIR, exist_ok=True)
IMG_SIZE     = (320, 320)
BATCH        = 64
NUM_WORKERS  = 4
SCORE_THRESH = 0.6                 # 사람 query objectness 임계(빡세게). 미만은 학습에서 완전 제외
TARGET_BOX   = [0.5, 0.5, 1.0, 1.0] # 크롭=전체프레임 (benchmark/추론과 동일 기준)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

tf = transforms.Compose([
    transforms.Resize(IMG_SIZE), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class ImgDS(Dataset):
    def __init__(self, csv): self.df = pd.read_csv(csv)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        try:    img = tf(Image.open(r['filepath']).convert('RGB'))
        except Exception: img = torch.zeros(3, *IMG_SIZE)
        y = torch.tensor([int(r['age']),int(r['gender']),int(r['up_color']),int(r['down_color'])], dtype=torch.long)
        return img, y, r['filepath']

# decoder가 query 임베딩을 반환하도록 (학습셀과 동일 구조)
class EmbedDecoder(DeformableTransformerDecoder):
    def forward(self, embed, refer_bbox, feats, shapes, bbox_head, score_head, pos_mlp, attn_mask=None, padding_mask=None):
        output = embed; dec_bboxes = []; refer_bbox = refer_bbox.sigmoid()
        for i, layer in enumerate(self.layers):
            output = layer(output, refer_bbox, feats, shapes, padding_mask, attn_mask, pos_mlp(refer_bbox))
            bbox    = bbox_head[i](output)
            refined = torch.sigmoid(bbox + inverse_sigmoid(refer_bbox))
            if i == self.eval_idx:
                dec_bboxes.append(refined); break
            refer_bbox = refined
        return torch.stack(dec_bboxes), output

class Extractor(nn.Module):
    def __init__(self):
        super().__init__()
        base = RTDETR('rtdetr-l.pt'); self.net = base.model
        self.dec = self.net.model[28]
        old = self.dec.decoder
        new = EmbedDecoder(hidden_dim=old.layers[0].self_attn.embed_dim,
                           decoder_layer=old.layers[0], num_layers=len(old.layers), eval_idx=old.eval_idx)
        new.load_state_dict(old.state_dict()); self.dec.decoder = new
        self.eval()
    def _backbone(self, x):
        y = []
        for m in self.net.model:
            if m.i == 28: return [y[j] for j in self.dec.f]
            if m.f != -1:
                x = y[m.f] if isinstance(m.f, int) else [x if j==-1 else y[j] for j in m.f]
            x = m(x); y.append(x if m.i in self.net.save else None)
    @torch.no_grad()
    def forward(self, x):
        with torch.cuda.amp.autocast():
            feats = self._backbone(x)
        feats = [f.float() for f in feats]
        feats, shapes = self.dec._get_encoder_input(feats)
        embed, refer_bbox, _, _ = self.dec._get_decoder_input(feats, shapes, None, None)
        dec_boxes, qemb = self.dec.decoder(
            embed, refer_bbox, feats, shapes,
            self.dec.dec_bbox_head, self.dec.dec_score_head, self.dec.query_pos_head, attn_mask=None)
        scores = self.dec.dec_score_head[self.dec.decoder.eval_idx](qemb)  # (bs,nq,nc)
        return dec_boxes[-1], qemb, scores

# ---- 이미 추출됐으면 스킵 ----
FORCE_REEXTRACT = False   # 다시 추출하려면 True
_dp = [f'{OUT_DIR}/train.pt', f'{OUT_DIR}/val.pt', f'{OUT_DIR}/kept_paths.json']
_need = FORCE_REEXTRACT or not all(os.path.exists(p) for p in _dp)
if (not _need) and ('boxes' not in torch.load(_dp[0], map_location='cpu')):
    _need = True   # 구버전 캐시(박스 없음) → 재추출
if _need:
    ext = Extractor().to(device)
    target = torch.tensor(TARGET_BOX, device=device)

    def run_split(csv, tag):
        dl = DataLoader(ImgDS(csv), batch_size=BATCH, num_workers=NUM_WORKERS, shuffle=False, pin_memory=True)
        X=[]; Y=[]; keep=[]; B=[]; sc_all=[]; n_skip=0
        for imgs, ys, paths in tqdm(dl, desc=f'extract {tag}'):
            imgs = imgs.to(device, non_blocking=True)
            boxes, qemb, scores = ext(imgs)
            boxes=boxes.float(); qemb=qemb.float()
            d  = (boxes - target.view(1,1,4)).abs().sum(-1)      # (bs,nq)
            q  = d.argmin(1)                                      # (bs,)
            bs = imgs.size(0); ar = torch.arange(bs, device=device)
            emb_q = qemb[ar, q]                                   # (bs,256)
            box_q = boxes[ar, q]                                  # (bs,4) cxcywh — 동결모델 pseudo-GT 박스
            sc_q  = scores[ar, q].float().sigmoid().max(-1).values  # (bs,) objectness 근사
            for b in range(bs):
                sc_all.append(float(sc_q[b]))
                if sc_q[b].item() < SCORE_THRESH:
                    n_skip += 1; continue
                X.append(emb_q[b].half().cpu()); Y.append(ys[b]); keep.append(paths[b]); B.append(box_q[b].cpu())
        X = torch.stack(X); Y = torch.stack(Y)
        torch.save({'X':X, 'Y':Y, 'paths':keep, 'boxes':torch.stack(B)}, f'{OUT_DIR}/{tag}.pt')
        sc = np.array(sc_all)
        print(f'[{tag}] keep={len(X)} skip={n_skip} ({n_skip/len(sc)*100:.1f}%) '
              f'score min/med/max={sc.min():.2f}/{np.median(sc):.2f}/{sc.max():.2f}')
        return keep

    kept = []
    kept += run_split(f'{UPAR_DIR}/train.csv', 'train')
    kept += run_split(f'{UPAR_DIR}/val.csv',   'val')
    with open(f'{OUT_DIR}/kept_paths.json', 'w') as f:
        json.dump(kept, f)
    print('Phase 1A 완료 →', OUT_DIR)

else:
    print('박스/pseudo-GT 이미 존재 → 스킵 (다시 만들려면 FORCE_REEXTRACT=True)')


In [ ]:
# ============================================================
# W&B 로깅 (선택) — 학습 곡선/지표 기록. 본인 키로 교체.
# ============================================================
!pip install -q wandb
import os, wandb
WANDB_API_KEY = "YOUR_WANDB_API_KEY"   # ← 본인 W&B API 키로 교체
os.environ["WANDB_API_KEY"] = WANDB_API_KEY
wandb.login(key=WANDB_API_KEY)
print("W&B 로그인 완료")

In [ ]:
# 학습: head-only(동결 RT-DETR) + RegionStem(원본 픽셀 가지)
!pip install ultralytics -q
import os, torch, torch.nn as nn, numpy as np, pandas as pd, random
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.ops import roi_align
from PIL import Image
from tqdm import tqdm
from ultralytics import RTDETR
from ultralytics.nn.modules.transformer import DeformableTransformerDecoder
from ultralytics.nn.modules.utils import inverse_sigmoid

device = 'cuda' if torch.cuda.is_available() else 'cpu'
CFG = dict(num_age=3, num_gender=2, num_up_color=11, num_down_color=11,
           epochs=30, batch=32, lr=3e-4, wd=1e-4, workers=2, S=320,
           crop_h=128, crop_w=64, region_dim=256, ls=0.1, warmup=2, clip=5.0)
ATTR_KEYS = ['age','gender','up_color','down_color']
ATTR_NUM  = {'age':3,'gender':2,'up_color':11,'down_color':11}
S = CFG['S']

# ---------- 데이터 (resize S + 실제 박스 + 0.7~0.9 필터) ----------
_norm = transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
_jit  = transforms.ColorJitter(0.2,0.2,0.1,0.0)

BOX_MAP = {}
for _tag in ['train','val']:
    _p = f'{CACHE}/{_tag}.pt'
    if os.path.exists(_p):
        _d = torch.load(_p, map_location='cpu')
        for _pth,_bx in zip(_d['paths'], _d['boxes']):
            BOX_MAP[os.path.splitext(os.path.basename(_pth))[0]] = _bx
assert BOX_MAP, '박스 캐시 없음 (query_cache_v0_3/train.pt). Phase1A 먼저 실행.'
keep = {k for k,v in BOX_MAP.items() if 0.7 <= float(v[2])*float(v[3]) <= 0.9}
print('0.7~0.9 박스:', len(keep))

class AttrDS(Dataset):
    def __init__(self, csv, train):
        df = pd.read_csv(csv)
        df = df[df['filepath'].map(lambda p: os.path.splitext(os.path.basename(p))[0] in keep)].reset_index(drop=True)
        self.df = df; self.train = train
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        key = os.path.splitext(os.path.basename(r['filepath']))[0]
        try:
            pil = Image.open(r['filepath']).convert('RGB').resize((S, S))
        except Exception:
            return torch.zeros(3, S, S), torch.tensor([-1,-1,-1,-1], dtype=torch.long), torch.tensor([0.5,0.5,1.,1.])
        box = BOX_MAP[key];  box = box if isinstance(box, torch.Tensor) else torch.tensor(box, dtype=torch.float32)
        box = box.float().clone()
        if self.train:
            pil = _jit(pil)
            if random.random() < 0.5:
                pil = pil.transpose(Image.FLIP_LEFT_RIGHT); box[0] = 1.0 - box[0]
        img = _norm(transforms.functional.to_tensor(pil))
        up, dn = int(r['up_color']), int(r['down_color'])
        up = up if 0 <= up < CFG['num_up_color'] else -1
        dn = dn if 0 <= dn < CFG['num_down_color'] else -1
        y = torch.tensor([int(r['age']), int(r['gender']), up, dn], dtype=torch.long)
        return img, y, box

tr = DataLoader(AttrDS(f'{UPAR_DIR}/train.csv', True),  batch_size=CFG['batch'], shuffle=True,
                num_workers=CFG['workers'], pin_memory=True, drop_last=True)
va = DataLoader(AttrDS(f'{UPAR_DIR}/val.csv',   False), batch_size=CFG['batch'], shuffle=False,
                num_workers=CFG['workers'], pin_memory=True)
print('train', len(tr.dataset), 'val', len(va.dataset))

# 크롭 이미지가 실제 디스크에 있는지 확인 (런타임 재시작 시 /content 휘발)
_miss = [fp for fp in tr.dataset.df['filepath'].head(30).tolist() if not os.path.exists(fp)]
if _miss:
    raise FileNotFoundError('크롭 이미지가 디스크에 없습니다 (예: ' + _miss[0] + '). 이 런타임의 /content/dataset 가 비어있습니다. 메인 노트북의 데이터 추출 셀을 이 런타임에서 먼저 실행해 크롭을 /content 에 풀어주세요.')

# ---------- 동결 RT-DETR backbone (dec_boxes + query 임베딩 반환) ----------
class EmbedDecoder(DeformableTransformerDecoder):
    def forward(self, embed, refer_bbox, feats, shapes, bbox_head, score_head, pos_mlp, attn_mask=None, padding_mask=None):
        output = embed; dec_bboxes = []; refer_bbox = refer_bbox.sigmoid()
        for i, layer in enumerate(self.layers):
            output = layer(output, refer_bbox, feats, shapes, padding_mask, attn_mask, pos_mlp(refer_bbox))
            bbox = bbox_head[i](output); refined = torch.sigmoid(bbox + inverse_sigmoid(refer_bbox))
            if i == self.eval_idx: dec_bboxes.append(refined); break
            refer_bbox = refined
        return torch.stack(dec_bboxes), output

class Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        base = RTDETR('rtdetr-l.pt'); self.net = base.model
        self.dec = self.net.model[28]
        old = self.dec.decoder
        new = EmbedDecoder(hidden_dim=old.layers[0].self_attn.embed_dim, decoder_layer=old.layers[0],
                           num_layers=len(old.layers), eval_idx=old.eval_idx)
        new.load_state_dict(old.state_dict()); self.dec.decoder = new
        self.eval()
        for p in self.parameters(): p.requires_grad = False
    def _feats(self, x):
        y = []
        for m in self.net.model:
            if m.i == 28: return [y[j] for j in self.dec.f]
            if m.f != -1: x = y[m.f] if isinstance(m.f, int) else [x if j==-1 else y[j] for j in m.f]
            x = m(x); y.append(x if m.i in self.net.save else None)
    @torch.no_grad()
    def forward(self, x):
        with torch.cuda.amp.autocast():
            feats = self._feats(x)
        feats = [f.float() for f in feats]
        feats, shapes = self.dec._get_encoder_input(feats)
        embed, refer_bbox, _, _ = self.dec._get_decoder_input(feats, shapes, None, None)
        dec_boxes, qemb = self.dec.decoder(embed, refer_bbox, feats, shapes,
                                           self.dec.dec_bbox_head, self.dec.dec_score_head,
                                           self.dec.query_pos_head, attn_mask=None)
        scores = self.dec.dec_score_head[self.dec.decoder.eval_idx](qemb)
        return dec_boxes[-1].float(), qemb.float(), scores.float()

# ---------- 추가: 원본 픽셀 가지 + 헤드 ----------
class RegionStem(nn.Module):
    def __init__(self, d=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,32,3,2,1),  nn.BatchNorm2d(32),  nn.ReLU(inplace=True),   # 128x64->64x32
            nn.Conv2d(32,64,3,2,1), nn.BatchNorm2d(64),  nn.ReLU(inplace=True),   # ->32x16
            nn.Conv2d(64,128,3,2,1),nn.BatchNorm2d(128), nn.ReLU(inplace=True),   # ->16x8
            nn.Conv2d(128,d,3,2,1), nn.BatchNorm2d(d),   nn.ReLU(inplace=True),   # ->8x4
            nn.AdaptiveAvgPool2d(1))
    def forward(self, x): return self.net(x).flatten(1)

class AttrHead(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(in_dim,1024), nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(0.3),
                                   nn.Linear(1024,512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3))
        self.heads = nn.ModuleDict({k: nn.Linear(512, ATTR_NUM[k]) for k in ATTR_KEYS})
    def forward(self, f):
        h = self.trunk(f); return {k: hd(h) for k, hd in self.heads.items()}

backbone = Backbone().to(device)
stem     = RegionStem(CFG['region_dim']).to(device)
head     = AttrHead(256 + CFG['region_dim']).to(device)

def cxcywh_to_xyxy(b):
    x,y,w,h = b.unbind(-1)
    return torch.stack([x-w/2, y-h/2, x+w/2, y+h/2], -1)

crit = {k: nn.CrossEntropyLoss(label_smoothing=CFG['ls'], ignore_index=-1) for k in ATTR_KEYS}
params = list(stem.parameters()) + list(head.parameters())
opt = torch.optim.AdamW(params, lr=CFG['lr'], weight_decay=CFG['wd'])
_w = CFG['warmup']
sch = torch.optim.lr_scheduler.SequentialLR(opt,
        [torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1, total_iters=_w),
         torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG['epochs']-_w)], milestones=[_w])

def run(dl, train):
    (stem.train(), head.train()) if train else (stem.eval(), head.eval())
    cor = {k:0 for k in ATTR_KEYS}; val = {k:0 for k in ATTR_KEYS}; tot=0.0; nb=0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, ys, gtb in tqdm(dl, leave=False):
            imgs = imgs.to(device, non_blocking=True); ys = ys.to(device); gtb = gtb.to(device)
            dec_boxes, qemb, _ = backbone(imgs)                       # 동결
            ba = torch.arange(imgs.size(0), device=device)
            qi = (dec_boxes - gtb.view(-1,1,4)).abs().sum(-1).argmin(1)
            q_sel = qemb[ba, qi]                                      # (bs,256)
            xyxy = (cxcywh_to_xyxy(gtb) * S).clamp(0, S)              # 픽셀 좌표
            roi = torch.cat([ba.view(-1,1).float(), xyxy], 1)        # (bs,5)
            crop = roi_align(imgs.float(), roi, output_size=(CFG['crop_h'], CFG['crop_w']), aligned=True)
            region = stem(crop)                                       # (bs,256)
            out = head(torch.cat([q_sel, region], 1))
            loss = sum(crit[k](out[k], ys[:,i]) for i,k in enumerate(ATTR_KEYS))
            if train:
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(params, CFG['clip']); opt.step()
            for i,k in enumerate(ATTR_KEYS):
                vm = ys[:,i] >= 0
                cor[k] += ((out[k].argmax(1)==ys[:,i]) & vm).sum().item(); val[k] += int(vm.sum().item())
            tot += float(loss.item()); nb += 1
    return tot/max(nb,1), {k: cor[k]/max(val[k],1) for k in ATTR_KEYS}

best = 0.0
for ep in range(1, CFG['epochs']+1):
    trl, tac = run(tr, True)
    vll, vac = run(va, False); sch.step()
    m = sum(vac.values())/4
    print(f"[{ep:02d}/{CFG['epochs']}] val_mean={m*100:.2f}% | age={vac['age']*100:.1f} gen={vac['gender']*100:.1f} "
          f"up={vac['up_color']*100:.1f} down={vac['down_color']*100:.1f} | trL={trl:.3f} vaL={vll:.3f} "
          f"trM={sum(tac.values())/4*100:.1f}")
    ckpt = {'stem':stem.state_dict(), 'head':head.state_dict(), 'cfg':CFG}
    torch.save(ckpt, f'{CKPT_DIR}/last.pt')
    if m > best:
        best = m; torch.save(ckpt, f'{CKPT_DIR}/best.pt'); print(f'  -> best {m*100:.2f}%')
print('완료 best mean =', round(best*100,2), '%')

## 추론 (장면 → RT-DETR 탐지 → query + 원본픽셀 조회 → 속성)
단일 모델: 동결 RT-DETR가 탐지/query 제공, 같은 장면에서 박스 픽셀을 crop해 RegionStem으로 조회.

In [ ]:
# ============================================================
# 추론용 테스트 데이터 준비 (benchmark / video 공통)
# ============================================================
import os
DATASET_DIR = '/content/drive/MyDrive/Workspace/RT-DETR_ReID/dataset'

# 1) 성능 테스트셋: val.csv (학습 제외 held-out, 라벨 포함)
val_csv = f'{DATASET_DIR}/UPAR/val.csv'
print('성능 테스트셋(val.csv):', os.path.exists(val_csv))
if os.path.exists(val_csv):
    import pandas as pd
    print('  테스트 장수:', len(pd.read_csv(val_csv)))

# 2) 데모 영상 3종 다운로드 (보행자/사람 등장 샘플)
DEMO_DIR = '/content/demo'
os.makedirs(DEMO_DIR, exist_ok=True)
GH_BASE = 'https://github.com/intel-iot-devkit/sample-videos/raw/master'
videos = {
    'people-detection.mp4':            f'{GH_BASE}/people-detection.mp4',
    'store-aisle-detection.mp4':       f'{GH_BASE}/store-aisle-detection.mp4',
    'one-by-one-person-detection.mp4': f'{GH_BASE}/one-by-one-person-detection.mp4',
}
for name, url in videos.items():
    out = f'{DEMO_DIR}/{name}'
    if not os.path.exists(out):
        print(f'다운로드: {name}')
        !wget -q -O "{out}" "{url}"
    if os.path.exists(out):
        ok = os.path.getsize(out) > 10000
        print(f'  {name}: {"OK" if ok else "실패"} ({os.path.getsize(out)/1e6:.1f}MB)')
    else:
        print(f'  {name}: 없음')

print('\n데모 영상 폴더:', DEMO_DIR, '->', os.listdir(DEMO_DIR))
print('\n준비 완료. 추론 셀에서 RUN_MODE 선택: benchmark / video / webcam')


In [ ]:
# -*- coding: utf-8 -*-
"""
RT-DETR_ReID 추론 (region-branch: 원본 RT-DETR 탐지 + RegionStem 원본픽셀 조회 + 속성헤드)
- 탐지/bbox: 원본 RT-DETR (검증됨)
- 속성: query 임베딩 + 박스 원본픽셀(RegionStem) concat → 헤드
모드: benchmark(성능) | video(데모영상) | webcam(실시간)  — 기존 추론과 동일 인터페이스
"""

# =============================================================================
# USER SETTINGS
# =============================================================================
RUN_MODE     = "video"   # "benchmark" | "image" | "video" | "webcam"
WEIGHTS      = "/content/drive/MyDrive/Workspace/RT-DETR_ReID/checkpoints/region_attr/best.pt"
BASE_WEIGHTS = "rtdetr-l.pt"
SOURCE       = "/content/demo"   # video 모드 입력 (폴더 = 안의 영상 전부)
TEST_CSV     = "/content/drive/MyDrive/Workspace/RT-DETR_ReID/dataset/UPAR/val.csv"  # benchmark 입력
BENCH_LIMIT  = 3000      # benchmark 평가 장수 (None=전체)
SAVE_DIR     = "/content/drive/MyDrive/Workspace/RT-DETR_ReID/infer_results"

IMGSZ      = 320
CONF_THRES = 0.35     # 원본 RT-DETR person 탐지 임계값
CLASS_ID   = 0        # COCO person
CROP_H, CROP_W = 128, 64   # RegionStem 입력(학습과 동일)

# Colab 웹캠
COLAB_WIDTH, COLAB_HEIGHT, COLAB_FPS = 640, 480, 10
SAVE_WEBCAM_VIDEO = True

# 로그
SAVE_LOG_CSV  = True
LOG_EVERY_SEC = 1.0
LOG_CSV_PATH  = "/content/drive/MyDrive/Workspace/RT-DETR_ReID/infer_results/colab_webcam_log.csv"

# ByteTrack
USE_BYTETRACK     = True
TRACK_HIGH_THRESH = 0.35
TRACK_LOW_THRESH  = 0.35
NEW_TRACK_THRESH  = 0.35
TRACK_BUFFER      = 30
MATCH_THRESH      = 0.80
FUSE_SCORE        = True

SHOW_FOLDER_RESULTS = False

# ── 특정 속성 선별 피플 카운터 ──
# None=전체 카운팅. dict=조건(AND) 통과한 고유 인물만 카운팅.
#   예) {"gender":"female"}                         여성만
#       {"up_color":"red"}                          빨간 상의만
#       {"gender":"male","up_color":"black"}        검은 상의 남성만
ATTR_FILTER = None

# 학습 라벨 순서와 반드시 동일 (11색)
AGE_NAMES        = ["young", "adult", "old"]
GENDER_NAMES     = ["male", "female"]
UP_COLOR_NAMES   = ["black", "blue", "brown", "green", "grey", "orange", "pink", "purple", "red", "white", "yellow"]
DOWN_COLOR_NAMES = ["black", "blue", "brown", "green", "grey", "orange", "pink", "purple", "red", "white", "yellow"]


# =============================================================================
# CODE
# =============================================================================
try:
    import ultralytics  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)

import base64, csv, os, time
from datetime import datetime
from pathlib import Path
from types import SimpleNamespace
from collections import defaultdict, Counter

import cv2
import numpy as np
import torch
import torch.nn as nn
from torchvision.ops import roi_align
from ultralytics import RTDETR
from ultralytics.nn.modules.transformer import DeformableTransformerDecoder
from ultralytics.nn.modules.utils import inverse_sigmoid

ATTR_KEYS = ["age", "gender", "up_color", "down_color"]
NUM_CLASSES = {"age": len(AGE_NAMES), "gender": len(GENDER_NAMES),
               "up_color": len(UP_COLOR_NAMES), "down_color": len(DOWN_COLOR_NAMES)}
LABEL_NAMES = {"age": AGE_NAMES, "gender": GENDER_NAMES,
               "up_color": UP_COLOR_NAMES, "down_color": DOWN_COLOR_NAMES}
LOG_FIELDNAMES = ["wall_time","elapsed_sec","frame_idx","det_idx","track_id",
                  "score","class_id","x1","y1","x2","y2",
                  "age","gender","up_color","down_color"]
IMAGE_EXTS = {".jpg",".jpeg",".png",".bmp",".webp"}
VIDEO_EXTS = {".mp4",".avi",".mov",".mkv",".wmv"}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DETECTOR = None   # 원본 RT-DETR (전역)


# ───────────────────────── region-branch 모델 ─────────────────────────
class EmbedDecoder(DeformableTransformerDecoder):
    def forward(self, embed, refer_bbox, feats, shapes, bbox_head, score_head, pos_mlp, attn_mask=None, padding_mask=None):
        output = embed; dec_bboxes = []; refer_bbox = refer_bbox.sigmoid()
        for i, layer in enumerate(self.layers):
            output = layer(output, refer_bbox, feats, shapes, padding_mask, attn_mask, pos_mlp(refer_bbox))
            bbox = bbox_head[i](output); refined = torch.sigmoid(bbox + inverse_sigmoid(refer_bbox))
            if i == self.eval_idx:
                dec_bboxes.append(refined); break
            refer_bbox = refined
        return torch.stack(dec_bboxes), output

class Backbone(nn.Module):
    """동결 RT-DETR → dec_boxes(cxcywh norm), query 임베딩, score."""
    def __init__(self, base_weights=BASE_WEIGHTS):
        super().__init__()
        base = RTDETR(base_weights); self.net = base.model
        self.dec = self.net.model[28]
        old = self.dec.decoder
        new = EmbedDecoder(hidden_dim=old.layers[0].self_attn.embed_dim, decoder_layer=old.layers[0],
                           num_layers=len(old.layers), eval_idx=old.eval_idx)
        new.load_state_dict(old.state_dict()); self.dec.decoder = new
        self.eval()
    def _feats(self, x):
        y = []
        for m in self.net.model:
            if m.i == 28: return [y[j] for j in self.dec.f]
            if m.f != -1: x = y[m.f] if isinstance(m.f, int) else [x if j==-1 else y[j] for j in m.f]
            x = m(x); y.append(x if m.i in self.net.save else None)
    @torch.no_grad()
    def forward(self, x):
        with torch.cuda.amp.autocast():
            feats = self._feats(x)
        feats = [f.float() for f in feats]
        feats, shapes = self.dec._get_encoder_input(feats)
        embed, refer_bbox, _, _ = self.dec._get_decoder_input(feats, shapes, None, None)
        dec_boxes, qemb = self.dec.decoder(embed, refer_bbox, feats, shapes,
                                           self.dec.dec_bbox_head, self.dec.dec_score_head,
                                           self.dec.query_pos_head, attn_mask=None)
        return dec_boxes[-1].float(), qemb.float()

class RegionStem(nn.Module):
    def __init__(self, d=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,32,3,2,1),  nn.BatchNorm2d(32),  nn.ReLU(inplace=True),
            nn.Conv2d(32,64,3,2,1), nn.BatchNorm2d(64),  nn.ReLU(inplace=True),
            nn.Conv2d(64,128,3,2,1),nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128,d,3,2,1), nn.BatchNorm2d(d),   nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1))
    def forward(self, x): return self.net(x).flatten(1)

class AttrHead(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(in_dim,1024), nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(0.3),
                                   nn.Linear(1024,512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3))
        self.heads = nn.ModuleDict({k: nn.Linear(512, NUM_CLASSES[k]) for k in ATTR_KEYS})
    def forward(self, f):
        h = self.trunk(f); return {k: hd(h) for k, hd in self.heads.items()}

def cxcywh_to_xyxy(b):
    x, y, w, h = b.unbind(-1)
    return torch.stack([x-w/2, y-h/2, x+w/2, y+h/2], -1)

class RegionAttrModel:
    """backbone(동결) + stem + head 묶음. stem/head만 학습 가중치 로드."""
    def __init__(self, base_weights=BASE_WEIGHTS):
        self.backbone = Backbone(base_weights).to(DEVICE).eval()
        self.stem = RegionStem(256).to(DEVICE).eval()
        self.head = AttrHead(256 + 256).to(DEVICE).eval()
    def load(self, weights):
        ck = torch.load(weights, map_location=DEVICE)
        self.stem.load_state_dict(ck["stem"]); self.head.load_state_dict(ck["head"])
        print("[OK] region 속성 가중치 정상 로드 (stem+head)")


def preprocess(frame_bgr):
    resized = cv2.resize(frame_bgr, (IMGSZ, IMGSZ), interpolation=cv2.INTER_LINEAR)
    rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
    tensor = torch.from_numpy(rgb).permute(2, 0, 1).float() / 255.0
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    tensor = (tensor - mean) / std
    return tensor.unsqueeze(0).to(DEVICE)


def _attrs_from(model, x, q_idx, box_xyxy_norm):
    """선택된 query + 박스영역 원본픽셀(roi) → 속성 logits dict (1샘플)."""
    q_sel = model._qemb[0, q_idx].unsqueeze(0)                          # (1,256)
    xyxy = (box_xyxy_norm * IMGSZ).clamp(0, IMGSZ).view(1, 4)
    roi = torch.cat([torch.zeros(1, 1, device=DEVICE), xyxy], 1)        # (1,5)
    crop = roi_align(x.float(), roi, output_size=(CROP_H, CROP_W), aligned=True)
    region = model.stem(crop)                                          # (1,256)
    return model.head(torch.cat([q_sel, region], 1))


# ───────────────────────── 추론 (region-branch) ─────────────────────────
@torch.no_grad()
def infer_frame(model, frame_bgr, conf_thres=CONF_THRES):
    # 1) 원본 RT-DETR로 사람 탐지
    det = DETECTOR(frame_bgr, conf=conf_thres, verbose=False)[0]
    if det.boxes is None or len(det.boxes) == 0:
        return []
    person_boxes = []
    for box in det.boxes:
        if int(box.cls[0]) != CLASS_ID:
            continue
        person_boxes.append({"xyxy": box.xyxy[0].cpu().numpy(),
                             "cxcywh": box.xywhn[0].cpu().numpy(),
                             "conf": float(box.conf[0])})
    if not person_boxes:
        return []

    # 2) backbone 1회: dec_boxes + query 임베딩
    x = preprocess(frame_bgr)
    q_boxes, qemb = model.backbone(x)
    q_boxes = q_boxes[0]; model._qemb = qemb

    # 3) 사람마다: 가까운 query 매칭 + 박스영역 원본픽셀 조회 → 속성
    out = []
    for pb in person_boxes:
        gt = torch.tensor(pb["cxcywh"], device=DEVICE, dtype=q_boxes.dtype)
        q  = (q_boxes - gt.unsqueeze(0)).abs().sum(dim=1).argmin().item()
        box_xyxy = cxcywh_to_xyxy(gt)
        logits = _attrs_from(model, x, q, box_xyxy)
        attrs = {k: LABEL_NAMES[k][logits[k].argmax(1).item()] for k in ATTR_KEYS}
        x1, y1, x2, y2 = pb["xyxy"].astype(int).tolist()
        out.append({"box": [x1, y1, x2, y2], "score": pb["conf"], "class_id": CLASS_ID, "attrs": attrs})
    return out


def passes_filter(attrs):
    if not ATTR_FILTER:
        return True
    for k, v in ATTR_FILTER.items():
        if attrs.get(k) != v:
            return False
    return True

def filter_label():
    """오버레이/로그용 현재 카운팅 대상 설명."""
    if not ATTR_FILTER:
        return "ALL people"
    return ", ".join(f"{k}={v}" for k, v in ATTR_FILTER.items())


# ───────────────────────── 피플 카운터 ─────────────────────────
class PeopleCounter:
    """track_id 기준 고유 인물 누적 카운터 (속성 필터 선별).
    영상/웹캠: 필터를 통과한 적이 있는 고유 track_id 수를 누적한다."""
    def __init__(self):
        self.counted = set()      # 필터 통과한 고유 track_id
        self.seen = set()         # 탐지된 전체 고유 track_id
    def update(self, results):
        """투표(vote_attrs) 적용된 결과를 받아 누적. 이번 프레임 매칭 인원 수 반환."""
        in_frame = 0
        for r in results:
            tid = r.get("track_id", -1)
            if tid >= 0:
                self.seen.add(tid)
            if passes_filter(r["attrs"]):
                in_frame += 1
                if tid >= 0:
                    self.counted.add(tid)
        return in_frame
    @property
    def total(self):
        return len(self.counted)
    @property
    def total_seen(self):
        return len(self.seen)


# ───────────────────────── ByteTrack ─────────────────────────
class TrackerDetections:
    def __init__(self, results):
        self.results = results
        if results:
            xyxy = np.asarray([r["box"] for r in results], dtype=np.float32)
            conf = np.asarray([r["score"] for r in results], dtype=np.float32)
            cls  = np.asarray([r["class_id"] for r in results], dtype=np.float32)
        else:
            xyxy = np.zeros((0,4),dtype=np.float32); conf=np.zeros((0,),dtype=np.float32); cls=np.zeros((0,),dtype=np.float32)
        xywh = xyxy.copy()
        xywh[:,2] = xyxy[:,2]-xyxy[:,0]; xywh[:,3] = xyxy[:,3]-xyxy[:,1]
        xywh[:,0] = xyxy[:,0]+xywh[:,2]/2; xywh[:,1] = xyxy[:,1]+xywh[:,3]/2
        self.xyxy=xyxy; self.xywh=xywh; self.conf=conf; self.cls=cls
    def __len__(self): return len(self.conf)
    def __getitem__(self, idx):
        if isinstance(idx,np.ndarray) and idx.dtype==bool:
            sel=[r for r,k in zip(self.results,idx) if k]
        elif isinstance(idx,(list,tuple,np.ndarray)):
            sel=[self.results[int(i)] for i in np.asarray(idx).tolist()]
        else:
            sel=[self.results[int(idx)]]
        return TrackerDetections(sel)

def create_bytetrack_tracker():
    if not USE_BYTETRACK: return None
    try:
        from ultralytics.trackers.byte_tracker import BYTETracker
    except Exception as e:
        print(f"[TRACK] ByteTrack 사용불가: {e}"); return None
    args = SimpleNamespace(track_high_thresh=TRACK_HIGH_THRESH, track_low_thresh=TRACK_LOW_THRESH,
                           new_track_thresh=NEW_TRACK_THRESH, track_buffer=TRACK_BUFFER,
                           match_thresh=MATCH_THRESH, fuse_score=FUSE_SCORE)
    try:
        tracker = BYTETracker(args)
    except TypeError:
        tracker = BYTETracker(args, frame_rate=COLAB_FPS)
    print("[TRACK] ByteTrack enabled")
    return tracker

def apply_bytetrack(tracker, results, frame, keep_raw_if_empty=False):
    if tracker is None:
        for r in results: r["track_id"]=-1
        return results
    tracked = tracker.update(TrackerDetections(results), img=frame)
    out=[]
    for row in tracked:
        if len(row)<8: continue
        x1,y1,x2,y2=[int(v) for v in row[:4]]
        tid=int(row[4]); score=float(row[5]); cls=int(row[6]); di=int(row[7])
        if 0<=di<len(results):
            item=dict(results[di])
            item["box"]=[x1,y1,x2,y2]; item["track_id"]=tid; item["score"]=score
            out.append(item)
    if keep_raw_if_empty and results and not out:
        for r in results: r["track_id"]=-1
        return results
    return out


# ──────────────── 트랙별 속성 누적 투표 (temporal voting) ────────────────
def new_track_memory():
    return defaultdict(lambda: {k: Counter() for k in ATTR_KEYS})
def vote_attrs(memory, results):
    for r in results:
        tid = r.get("track_id", -1)
        if tid < 0:
            continue
        for k in ATTR_KEYS:
            memory[tid][k][r["attrs"][k]] += 1
        r["attrs"] = {k: memory[tid][k].most_common(1)[0][0] for k in ATTR_KEYS}
    return results


# ───────────────────────── 시각화 ─────────────────────────
def draw_results(frame, results, total=None):
    """필터 통과 인물만 박스/라벨. total(고유 누적)이 주어지면 카운터 오버레이 표시."""
    in_frame = 0
    for r in results:
        match = passes_filter(r["attrs"])
        x1,y1,x2,y2 = r["box"]
        a = r["attrs"]
        # 매칭은 초록 박스, 비매칭(필터 활성 시)은 회색 얇은 박스로 구분
        if not match:
            if ATTR_FILTER:
                cv2.rectangle(frame,(x1,y1),(x2,y2),(150,150,150),1)
            continue
        in_frame += 1
        tid = f"ID {r['track_id']} | " if r.get("track_id",-1)>=0 else ""
        label = f"{tid}{r['score']:.2f} | {a['gender']} {a['age']} {a['up_color']}/{a['down_color']}"
        cv2.rectangle(frame,(x1,y1),(x2,y2),(0,220,0),2)
        (tw,th),_ = cv2.getTextSize(label,cv2.FONT_HERSHEY_SIMPLEX,0.5,1)
        cv2.rectangle(frame,(x1,max(0,y1-th-8)),(min(x1+tw+6,frame.shape[1]-1),y1),(0,220,0),-1)
        cv2.putText(frame,label,(x1+3,max(12,y1-5)),cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,0,0),1,cv2.LINE_AA)
    # 상단 오버레이: 필터 / 현재 프레임 / 누적 고유 카운트
    cv2.putText(frame, f"Filter: {filter_label()}", (10,26), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,255), 2, cv2.LINE_AA)
    cv2.putText(frame, f"In frame: {in_frame}", (10,52), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,200,0), 2, cv2.LINE_AA)
    if total is not None:
        cv2.putText(frame, f"TOTAL counted: {total}", (10,82), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,0,255), 2, cv2.LINE_AA)
    return frame


# ───────────────────────── 파일/영상 처리 ─────────────────────────
def iter_media_files(source):
    source = Path(source)
    if source.is_file():
        yield source; return
    for p in sorted(source.rglob("*")):
        if p.suffix.lower() in IMAGE_EXTS or p.suffix.lower() in VIDEO_EXTS:
            yield p

def run_image(model, path):
    """이미지: 파일당 로그 1줄. matched = 필터 통과 인원, total = 전체 사람 수."""
    frame = cv2.imread(str(path))
    if frame is None:
        print(f"[SKIP] {path}"); return
    results = infer_frame(model, frame, conf_thres=CONF_THRES)
    for r in results: r["track_id"]=-1
    matched = sum(1 for r in results if passes_filter(r["attrs"]))
    vis = draw_results(frame, results)   # total 미표시(이미지엔 누적 개념 없음)
    out_path = Path(SAVE_DIR)/f"{path.stem}_pred.jpg"
    cv2.imwrite(str(out_path), vis)
    # 파일당 로그 1줄 (+ CSV append)
    print(f"[IMAGE] {path.name}: people={len(results)}  matched({filter_label()})={matched}  -> {out_path}")
    _append_count_log(Path(SAVE_DIR)/"image_counts.csv",
                      {"file": path.name, "people": len(results), "matched": matched, "filter": filter_label()})
    if SHOW_FOLDER_RESULTS and is_colab():
        from google.colab.patches import cv2_imshow
        cv2_imshow(vis)

def _append_count_log(csv_path, row):
    new = not Path(csv_path).exists()
    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys()))
        if new: w.writeheader()
        w.writerow(row)

def run_video(model, path):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        print(f"[SKIP] {path}"); return
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out_path = Path(SAVE_DIR)/f"{path.stem}_pred.mp4"
    writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (W,H))
    tracker = create_bytetrack_tracker()
    memory = new_track_memory()
    counter = PeopleCounter()
    n=0; t0=time.time()
    while True:
        ok, frame = cap.read()
        if not ok: break
        raw = infer_frame(model, frame, conf_thres=CONF_THRES)
        results = apply_bytetrack(tracker, raw, frame, keep_raw_if_empty=True)
        results = vote_attrs(memory, results)
        counter.update(results)                       # 고유 인물 누적
        writer.write(draw_results(frame, results, total=counter.total))
        n+=1
    cap.release(); writer.release()
    el = time.time()-t0
    print(f"[VIDEO] {path.name}: {n} frames | unique people={counter.total_seen} | "
          f"matched({filter_label()})={counter.total}  -> {out_path}  ({n/max(el,1e-6):.1f} FPS 처리)")
    _append_count_log(Path(SAVE_DIR)/"video_counts.csv",
                      {"file": path.name, "frames": n, "unique_people": counter.total_seen,
                       "matched": counter.total, "filter": filter_label(),
                       "proc_fps": round(n/max(el,1e-6),1)})

def run_folder(model):
    os.makedirs(SAVE_DIR, exist_ok=True)
    files = list(iter_media_files(SOURCE))
    if not files:
        print(f"파일 없음: {SOURCE}"); return
    for path in files:
        s = path.suffix.lower()
        if s in IMAGE_EXTS: run_image(model, path)
        elif s in VIDEO_EXTS: run_video(model, path)


# ───────────────────────── Colab 웹캠 ─────────────────────────
def is_colab():
    try:
        import google.colab; return True
    except Exception:
        return False

def js_to_bgr(data_url):
    _, enc = data_url.split(",",1)
    arr = np.frombuffer(base64.b64decode(enc), dtype=np.uint8)
    return cv2.imdecode(arr, cv2.IMREAD_COLOR)

def run_colab_webcam(model):
    if not is_colab():
        raise RuntimeError("webcam은 Colab에서만 가능")
    from IPython.display import Image, Javascript, display
    from google.colab.output import eval_js
    display(Javascript("""
    async function startCamera(w,h){
      if(window.__s)return true;
      const d=document.createElement('div');const b=document.createElement('button');
      b.textContent='Start camera';b.style.fontSize='16px';b.style.padding='8px 14px';
      const v=document.createElement('video');v.style.display='none';v.width=w;v.height=h;
      document.body.appendChild(d);d.appendChild(b);d.appendChild(v);
      const c=document.createElement('canvas');c.width=w;c.height=h;
      window.__v=v;window.__c=c;
      return await new Promise((res,rej)=>{b.onclick=async()=>{
        try{const s=await navigator.mediaDevices.getUserMedia({video:{width:w,height:h},audio:false});
        window.__s=s;v.srcObject=s;await v.play();b.disabled=true;res(true);}
        catch(e){rej(e.name);}};});
    }
    function captureFrame(){const v=window.__v,c=window.__c;
      c.getContext('2d').drawImage(v,0,0,c.width,c.height);return c.toDataURL('image/jpeg',0.85);}
    function stopCamera(){if(window.__s)window.__s.getTracks().forEach(t=>t.stop());return true;}
    """))
    print("[WEBCAM] Start camera 버튼 누르고 권한 허용")
    try:
        eval_js(f"startCamera({COLAB_WIDTH},{COLAB_HEIGHT})", timeout_sec=300)
    except TypeError:
        eval_js(f"startCamera({COLAB_WIDTH},{COLAB_HEIGHT})")

    writer=None; out_path=None
    if SAVE_WEBCAM_VIDEO:
        os.makedirs(SAVE_DIR, exist_ok=True)
        out_path=Path(SAVE_DIR)/"colab_webcam_pred.mp4"
        writer=cv2.VideoWriter(str(out_path),cv2.VideoWriter_fourcc(*"mp4v"),COLAB_FPS,(COLAB_WIDTH,COLAB_HEIGHT))
    tracker=create_bytetrack_tracker()
    memory=new_track_memory()
    counter=PeopleCounter()

    log_file=log_writer=None
    if SAVE_LOG_CSV:
        Path(LOG_CSV_PATH).parent.mkdir(parents=True, exist_ok=True)
        log_file=open(LOG_CSV_PATH,"a",newline="",encoding="utf-8")
        log_writer=csv.DictWriter(log_file,fieldnames=LOG_FIELDNAMES)
        if log_file.tell()==0: log_writer.writeheader()

    try:
        handle=None; t0=time.time(); last=-LOG_EVERY_SEC; fi=0
        while True:
            frame=js_to_bgr(eval_js("captureFrame()"))
            raw=infer_frame(model, frame, conf_thres=CONF_THRES)
            results=apply_bytetrack(tracker, raw, frame)
            results=vote_attrs(memory, results)
            counter.update(results)                       # 실시간 누적 고유 인물
            vis=draw_results(frame, results, total=counter.total)
            if writer is not None: writer.write(vis)
            el=time.time()-t0
            if log_writer is not None and el-last>=LOG_EVERY_SEC:
                wt=datetime.now().isoformat(timespec="seconds")
                if not results:
                    log_writer.writerow({"wall_time":wt,"elapsed_sec":round(el,3),"frame_idx":fi,
                        "det_idx":-1,"track_id":-1,"score":"","class_id":"","x1":"","y1":"","x2":"","y2":"",
                        "age":"none","gender":"none","up_color":"none","down_color":"none"})
                for di,r in enumerate(results):
                    x1,y1,x2,y2=r["box"]; a=r["attrs"]
                    log_writer.writerow({"wall_time":wt,"elapsed_sec":round(el,3),"frame_idx":fi,
                        "det_idx":di,"track_id":r.get("track_id",-1),"score":round(r["score"],5),
                        "class_id":r["class_id"],"x1":x1,"y1":y1,"x2":x2,"y2":y2,
                        "age":a["age"],"gender":a["gender"],"up_color":a["up_color"],"down_color":a["down_color"]})
                log_file.flush(); last=el
            ok,enc=cv2.imencode(".jpg",vis,[int(cv2.IMWRITE_JPEG_QUALITY),85])
            if ok:
                img=Image(data=enc.tobytes())
                if handle is None: handle=display(img, display_id=True)
                else: handle.update(img)
            fi+=1
    except KeyboardInterrupt:
        print("[WEBCAM] stopped")
    finally:
        eval_js("stopCamera()")
        print(f"[WEBCAM] unique people={counter.total_seen} | matched({filter_label()})={counter.total}")
        if writer is not None: writer.release(); print(f"[WEBCAM] saved {out_path}")
        if log_file is not None: log_file.close()


# ──────────────── 성능 테스트 (benchmark) ────────────────
@torch.no_grad()
def run_benchmark(model):
    """라벨 포함 테스트셋으로 속성별 정확도 측정 (크롭=사람 전체프레임 가정, 학습과 동일 입력)."""
    import pandas as pd
    from tqdm import tqdm
    df = pd.read_csv(TEST_CSV)
    if BENCH_LIMIT:
        df = df.sample(min(BENCH_LIMIT, len(df)), random_state=0).reset_index(drop=True)
    # 이미지 존재 가드: val.csv 경로는 /content/dataset(런타임 임시) → 추출 셀 선실행 필요
    _probe = [p for p in df["filepath"].head(50) if cv2.imread(str(p)) is not None]
    if not _probe:
        raise FileNotFoundError(
            "val.csv 이미지가 하나도 안 열립니다 (예: " + str(df["filepath"].iloc[0]) + "). "
            "이미지가 /content/dataset 에 없습니다. 상단의 '이미지 추출' 셀을 먼저 실행해 "
            "Drive zip → /content/dataset 으로 압축을 푼 뒤 벤치마크를 다시 실행하세요.")
    full_cxcywh = torch.tensor([0.5,0.5,1.0,1.0], device=DEVICE)
    full_xyxy   = torch.tensor([0.0,0.0,1.0,1.0], device=DEVICE)   # 크롭 전체영역
    correct = {k: 0 for k in ATTR_KEYS}; valid = {k: 0 for k in ATTR_KEYS}; n = 0
    miss = 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc="benchmark"):
        frame = cv2.imread(str(row["filepath"]))
        if frame is None:
            miss += 1; continue
        x = preprocess(frame)
        q_boxes, qemb = model.backbone(x); q_boxes = q_boxes[0]; model._qemb = qemb
        q = (q_boxes - full_cxcywh.unsqueeze(0)).abs().sum(dim=1).argmin().item()
        logits = _attrs_from(model, x, q, full_xyxy)
        gt = {k: int(row[k]) for k in ATTR_KEYS}
        for k in ATTR_KEYS:
            if gt[k] < 0 or gt[k] >= NUM_CLASSES[k]:
                continue
            valid[k] += 1
            correct[k] += int(logits[k].argmax(1).item() == gt[k])
        n += 1
    print(f"\n===== 성능 테스트 결과 ({n}장 평가, {miss}장 누락) =====")
    accs = []
    for k in ATTR_KEYS:
        acc = correct[k] / max(valid[k], 1); accs.append(acc)
        print(f"  {k:11s}: {acc*100:5.2f}%  (n={valid[k]})")
    print(f"  {'mean':11s}: {sum(accs)/len(accs)*100:5.2f}%")


# ───────────────────────── main ─────────────────────────
def main():
    global DETECTOR
    print(f"device: {DEVICE} | mode: {RUN_MODE}")
    model = RegionAttrModel(BASE_WEIGHTS)
    model.load(WEIGHTS)
    if RUN_MODE != "benchmark":
        DETECTOR = RTDETR(BASE_WEIGHTS)
        print("탐지용 vanilla RT-DETR 로드 완료")
    if RUN_MODE == "benchmark":
        run_benchmark(model)
    elif RUN_MODE in ("video", "image"):   # SOURCE의 확장자로 자동 분기 (폴더 혼재 가능)
        run_folder(model)
    elif RUN_MODE == "webcam":
        run_colab_webcam(model)
    else:
        raise ValueError('RUN_MODE: "benchmark" | "image" | "video" | "webcam"')

main()
